In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [3]:
# Part 1 — Load & Clean

cpi_df = pd.read_csv("fomc_etf_data/cpi_unemployment_etf_options.csv")
cpi_df["event_date"] = pd.to_datetime(cpi_df["event_date"])
cpi_df["entry_date"] = pd.to_datetime(cpi_df["entry_date"])
cpi_df["exit_date"]  = pd.to_datetime(cpi_df["exit_date"])

print(f"Raw rows         : {len(cpi_df):,}")

KEY    = ["ticker","event_date","event_type","entry_offset","exit_offset"]
before = len(cpi_df)
cpi_df = (cpi_df.sort_values("entry_straddle_mid", ascending=False)
                .drop_duplicates(subset=KEY, keep="first")
                .reset_index(drop=True))
print(f"After dedup      : {len(cpi_df):,} rows (removed {before-len(cpi_df):,})")

cpi_df = cpi_df[~((cpi_df["entry_offset"]==0) & (cpi_df["exit_offset"]==0))].copy()
print(f"After T0/T0 excl.: {len(cpi_df):,} rows")

print(f"\nDataset Overview:")
print(f"  Event types  : {sorted(cpi_df['event_type'].unique())}")
print(f"  Years        : {sorted(cpi_df['year'].unique())}")
print(f"  Tickers      : {cpi_df['ticker'].nunique()}  {sorted(cpi_df['ticker'].unique())}")
print(f"  Events       : {cpi_df.groupby('event_type')['event_date'].nunique().to_dict()}")
print(f"  Entry offsets: {sorted(cpi_df['entry_offset'].unique())}")
print(f"  Exit offsets : {sorted(cpi_df['exit_offset'].unique())}")
print(f"  Price source : {cpi_df['entry_price_source'].value_counts().to_dict()}")

print(f"\nMissing Data:")
key_cols = ["pnl_pct","entry_straddle_mid","exit_straddle_mid","entry_atm_iv","exit_atm_iv","stock_move"]
print(cpi_df[key_cols].isnull().sum().to_string())

print(f"\nRows by event type and ticker:")
print(cpi_df.groupby(["event_type","ticker"]).size().unstack(fill_value=0).to_string())

Raw rows         : 16,566
After dedup      : 16,566 rows (removed 0)
After T0/T0 excl.: 15,234 rows

Dataset Overview:
  Event types  : ['CPI', 'Unemployment']
  Years        : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Tickers      : 15  ['EEM', 'GLD', 'ITA', 'IWM', 'QQQ', 'SPY', 'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLV', 'XOP']
  Events       : {'CPI': 126, 'Unemployment': 119}
  Entry offsets: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0)]
  Exit offsets : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Price source : {'vwap+vwap': 15234}

Missing Data:
pnl_pct                0
entry_straddle_mid     0
exit_straddle_mid      0
entry_atm_iv           0
exit_atm_iv           35
stock_move             0

Rows by event type and ticker:
ticker        EEM   GLD  ITA  IWM   QQQ  SPY  TLT  XLB  XLE  XLF  XLI  XLK  XLP  XLV  XOP
event_type      

In [6]:
import plotly.graph_objects as go

BG    = "#ffffff"
PANEL = "#f8f8f8"
RED   = "#e8133a"
GREEN = "#2db82d"
GREY  = "#555555"
DG    = "#dddddd"
FONT  = "Garamond, 'EB Garamond', Georgia, serif"

e_offs_macro = [-3, -2, -1, 0]
x_offs_macro = [0, 1, 2, 3]

In [7]:
# Part 2 — Configuration Analysis Setup (run once)
e_offs_macro = [-3, -2, -1, 0]
x_offs_macro = [0, 1, 2, 3]

TICKER_META = {
    "SPY": "S&P 500",      "QQQ": "Nasdaq 100",
    "XLK": "Technology",   "XLF": "Financials",
    "XLE": "Energy",       "XLI": "Industrials",
    "XLV": "Health Care",  "XLB": "Materials",
    "XLP": "Consumer Staples", "GLD": "Gold",
    "TLT": "Long Treasuries",  "IWM": "Russell 2000",
    "EEM": "Emerging Markets", "XOP": "Oil & Gas E&P",
    "ITA": "Aerospace & Defense",
}

def plot_macro(event_type=None, ticker=None):
    sub = cpi_df.copy()
    if event_type: sub = sub[sub["event_type"] == event_type]
    if ticker:     sub = sub[sub["ticker"] == ticker]

    parts = []
    if event_type: parts.append(event_type)
    if ticker:     parts.append(f"{ticker} · {TICKER_META.get(ticker, ticker)}")
    if not parts:  parts.append("All Events · All ETFs")
    title = "Macro Straddle  ·  Avg Return per Combo  (" + "  |  ".join(parts) + ")"

    st = (sub.groupby(["entry_offset","exit_offset"])
             .agg(
                 n        = ("pnl_pct","count"),
                 avg_ret  = ("pnl_pct","mean"),
                 win_rate = ("pnl_pct", lambda x: (x>0).mean()),
             )
             .reset_index())
    idx_m = st.set_index(["entry_offset","exit_offset"])

    z, txt = [], []
    for e in e_offs_macro:
        rz, rt = [], []
        for x in x_offs_macro:
            if e == 0 and x == 0:
                rz.append(None); rt.append("T0/T0\nexcl.")
            elif (e, x) in idx_m.index:
                v  = idx_m.loc[(e,x), "avg_ret"]
                wr = idx_m.loc[(e,x), "win_rate"]
                n  = int(idx_m.loc[(e,x), "n"])
                rz.append(v)
                rt.append(f"{v:.2%}\nWR {wr:.0%}  n={n}")
            else:
                rz.append(None); rt.append("")
        z.append(rz); txt.append(rt)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=[f"Exit T{x:+d}" for x in x_offs_macro],
        y=[f"Entry T{e:+d}" for e in e_offs_macro],
        text=txt, texttemplate="%{text}",
        textfont=dict(family=FONT, size=12, color="#111111"),
        colorscale=[[0,RED],[0.45,"#f5a0aa"],[0.5,"#ffffff"],
                    [0.55,"#a0e6a0"],[1,GREEN]],
        zmid=0, showscale=True,
        colorbar=dict(tickfont=dict(family=FONT, color=GREY, size=11),
                      outlinecolor=DG, outlinewidth=1),
        hoverongaps=False,
        hovertemplate="Entry %{y} / Exit %{x}<br>%{text}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=title, font=dict(family=FONT, size=19, color="#111111"),
                   x=0.03, xanchor="left"),
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family=FONT, color="#111111", size=13),
        height=440, margin=dict(l=65, r=40, t=70, b=40),
        hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                        font=dict(family=FONT, color="#111111", size=13)),
    )
    fig.update_xaxes(side="top", tickfont=dict(family=FONT, color="#111111", size=13),
                     gridcolor=DG, linecolor=DG)
    fig.update_yaxes(autorange="reversed",
                     tickfont=dict(family=FONT, color="#111111", size=13),
                     gridcolor=DG, linecolor=DG)
    fig.show()

In [8]:
# Overall — all events, all ETFs
plot_macro()

H0: CPI

In [9]:
# By event type
plot_macro(event_type="CPI")
plot_macro(event_type="Unemployment")

In [11]:
# By event type + specific ETF
plot_macro(event_type="CPI",          ticker="TLT")
plot_macro(event_type="CPI",          ticker="GLD")
plot_macro(event_type="CPI",          ticker="XLF")
plot_macro(event_type="CPI",          ticker="XLK")
plot_macro(event_type="Unemployment", ticker="SPY")
plot_macro(event_type="Unemployment", ticker="IWM")
plot_macro(event_type="Unemployment", ticker="XLF")

In [16]:
# Part 3 — Walk-Forward Long/Short Strategy
# Signals on (event_type, ticker, entry_offset, exit_offset)

LONG_AVG_RET      =  0.05
LONG_MIN_N        =  500
SHORT_AVG_RET     = -0.05
SHORT_MIN_N       =  10
CAPITAL_PER_TRADE =  1_000
PORTFOLIO_SIZE    = 10_000

train = cpi_df[cpi_df["year"] <= 2021].copy()
test  = cpi_df[cpi_df["year"] >= 2022].copy()

print(f"Training set : {len(train):,} rows | years {sorted(train['year'].unique())}")
print(f"Test set     : {len(test):,}  rows | years {sorted(test['year'].unique())}")

combo_stats = (train.groupby(["event_type","ticker","entry_offset","exit_offset"])
                    .agg(
                        n        = ("pnl_pct","count"),
                        avg_ret  = ("pnl_pct","mean"),
                        med_ret  = ("pnl_pct","median"),
                        win_rate = ("pnl_pct", lambda x: (x>0).mean()),
                        std      = ("pnl_pct","std"),
                    )
                    .reset_index())

long_signals = combo_stats[
    (combo_stats["n"]       > LONG_MIN_N)   &
    (combo_stats["avg_ret"] > LONG_AVG_RET) &
    (combo_stats["med_ret"] > LONG_MED_RET)
].copy()
long_signals["side"] = "LONG"

short_signals = combo_stats[
    (combo_stats["n"]       > SHORT_MIN_N)   &
    (combo_stats["avg_ret"] < SHORT_AVG_RET) &
    (combo_stats["med_ret"] < SHORT_MED_RET)
].copy()
short_signals["side"] = "SHORT"

print(f"\nSignals from training (2016-2021):")
print(f"  Long  : {len(long_signals)}")
print(f"  Short : {len(short_signals)}")

print(f"\nLong signals:")
print(long_signals[["event_type","ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret", ascending=False)
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

print(f"\nShort signals:")
print(short_signals[["event_type","ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret")
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

long_keys  = set(zip(long_signals["event_type"],  long_signals["ticker"],
                     long_signals["entry_offset"], long_signals["exit_offset"]))
short_keys = set(zip(short_signals["event_type"],  short_signals["ticker"],
                     short_signals["entry_offset"], short_signals["exit_offset"]))

test_signals = test.copy()
test_signals["signal"] = None
test_signals.loc[test_signals.apply(
    lambda r: (r["event_type"],r["ticker"],r["entry_offset"],r["exit_offset"])
              in long_keys, axis=1), "signal"] = "LONG"
test_signals.loc[test_signals.apply(
    lambda r: (r["event_type"],r["ticker"],r["entry_offset"],r["exit_offset"])
              in short_keys, axis=1), "signal"] = "SHORT"

test_signals = test_signals[test_signals["signal"].notna()].copy()
test_signals["strategy_pnl"] = test_signals.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)

print(f"\n2022-2025 test trades : {len(test_signals)}")
print(f"  Long  : {(test_signals['signal']=='LONG').sum()}")
print(f"  Short : {(test_signals['signal']=='SHORT').sum()}")

def perf(returns, label):
    r   = pd.Series(returns).dropna()
    if len(r) == 0: print(f"\n  {label}: no trades"); return
    inv = len(r) * CAPITAL_PER_TRADE
    print(f"\n  {label}")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg return      : {r.mean():.2%}")
    print(f"    Median ret      : {r.median():.2%}")
    print(f"    Total PnL       : ${r.sum()*CAPITAL_PER_TRADE:,.2f}")
    print(f"    ROI             : {r.sum()*CAPITAL_PER_TRADE/PORTFOLIO_SIZE:.2%}")
    print(f"    Profit factor   : {r[r>0].sum()/abs(r[r<0].sum()):.2f}" if (r<0).any() else "    Profit factor   : inf")
    print(f"    Avg win/loss    : {r[r>0].mean()/abs(r[r<0].mean()):.2f}" if (r<0).any() else "    Avg win/loss    : inf")
    print(f"    Best trade      : ${r.max()*CAPITAL_PER_TRADE:,.2f}")
    print(f"    Worst trade     : ${r.min()*CAPITAL_PER_TRADE:,.2f}")

print(f"\nOut-of-Sample Performance (2022-2025):")
perf(test_signals["strategy_pnl"], "Combined")
perf(test_signals[test_signals["signal"]=="LONG"]["strategy_pnl"],  "Long only")
perf(test_signals[test_signals["signal"]=="SHORT"]["strategy_pnl"], "Short only")

print(f"\nBy Event Type:")
for et in ["CPI","Unemployment"]:
    perf(test_signals[test_signals["event_type"]==et]["strategy_pnl"], et)

print(f"\nBy Event Type x Signal:")
for et in ["CPI","Unemployment"]:
    for side in ["LONG","SHORT"]:
        sub = test_signals[(test_signals["event_type"]==et) & (test_signals["signal"]==side)]
        perf(sub["strategy_pnl"], f"{et} {side}")

strat = cpi_df.copy()
strat["signal"] = None
strat.loc[strat.apply(
    lambda r: (r["event_type"],r["ticker"],r["entry_offset"],r["exit_offset"])
              in long_keys, axis=1), "signal"] = "LONG"
strat.loc[strat.apply(
    lambda r: (r["event_type"],r["ticker"],r["entry_offset"],r["exit_offset"])
              in short_keys, axis=1), "signal"] = "SHORT"

strat = strat[strat["signal"].notna()].copy()
strat["strategy_pnl"] = strat.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)
strat["period"]     = strat["year"].apply(lambda y: "train" if y <= 2021 else "test")
strat["dollar_pnl"] = strat["strategy_pnl"] * CAPITAL_PER_TRADE
strat = strat.sort_values("exit_date").reset_index(drop=True)
strat["cum_dollar_pnl"]  = strat.groupby("period")["dollar_pnl"].cumsum()
strat["overall_cum_pnl"] = strat["dollar_pnl"].cumsum()
strat["peak"]            = strat["overall_cum_pnl"].cummax()
strat["drawdown"]        = strat["overall_cum_pnl"] - strat["peak"]

print(f"\nOverall Strategy Performance  (${CAPITAL_PER_TRADE:,} per trade)")
for period, grp in strat.groupby("period"):
    r   = grp["dollar_pnl"]
    inv = len(grp) * CAPITAL_PER_TRADE
    print(f"\n  {period.upper()} ({grp['year'].min()}-{grp['year'].max()})")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Total PnL       : ${r.sum():,.2f}")
    print(f"    ROI             : {r.sum()/PORTFOLIO_SIZE:.2%}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg per trade   : ${r.mean():,.2f}")
    print(f"    Profit factor   : {r[r>0].sum()/abs(r[r<0].sum()):.2f}" if (r<0).any() else "    Profit factor   : inf")
    print(f"    Best trade      : ${r.max():,.2f}")
    print(f"    Worst trade     : ${r.min():,.2f}")

r   = strat["dollar_pnl"]
inv = len(strat) * CAPITAL_PER_TRADE
print(f"\n  FULL PERIOD (2016-2025)")
print(f"    Trades          : {len(r)}")
print(f"    Capital deployed: ${inv:,.0f}")
print(f"    Total PnL       : ${r.sum():,.2f}")
print(f"    ROI             : {r.sum()/PORTFOLIO_SIZE:.2%}")
print(f"    Win rate        : {(r>0).mean():.1%}")
print(f"    Avg per trade   : ${r.mean():,.2f}")
print(f"    Max Drawdown    : ${strat['drawdown'].min():,.2f}")
print(f"    Profit factor   : {r[r>0].sum()/abs(r[r<0].sum()):.2f}" if (r<0).any() else "    Profit factor   : inf")
print(f"    Best trade      : ${r.max():,.2f}")
print(f"    Worst trade     : ${r.min():,.2f}")

fig = go.Figure()
for period, col, name in [("train", "#2E74B5", "Training 2016-2021"),
                           ("test",  GREEN,     "Test 2022-2025")]:
    grp = strat[strat["period"]==period]
    fig.add_trace(go.Scatter(
        x=grp["exit_date"], y=grp["cum_dollar_pnl"],
        mode="lines", name=name,
        line=dict(color=col, width=2),
        hovertemplate="%{x|%b %Y}<br>Cumulative PnL: $%{y:,.0f}<extra></extra>",
    ))

fig.add_vline(x=pd.Timestamp("2022-01-01").timestamp()*1000,
              line_color=DG, line_dash="dash",
              annotation_text="2022 test begins",
              annotation_font=dict(family=FONT, size=11, color=GREY))
fig.add_hline(y=0, line_color=DG, line_dash="dot")

fig.update_layout(
    title=dict(
        text=f"CPI & Unemployment Straddle  ·  Cumulative PnL  (${CAPITAL_PER_TRADE:,} per trade)",
        font=dict(family=FONT, size=19, color="#111111"),
        x=0.03, xanchor="left"),
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(family=FONT, color="#111111", size=13),
    height=460, margin=dict(l=80, r=40, t=70, b=60),
    legend=dict(bgcolor=BG, bordercolor=DG, borderwidth=1,
                font=dict(family=FONT, size=12)),
    hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                    font=dict(family=FONT, color="#111111", size=13)),
    xaxis=dict(gridcolor=DG, linecolor=DG,
               tickfont=dict(family=FONT, color="#333333")),
    yaxis=dict(title="Cumulative PnL ($)",
               tickprefix="$", tickformat=",.0f",
               gridcolor=DG, linecolor=DG, zerolinecolor=DG,
               tickfont=dict(family=FONT, color="#333333"),
               title_font=dict(family=FONT, color="#333333", size=12)),
)
fig.show()

Training set : 5,745 rows | years [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Test set     : 9,489  rows | years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Signals from training (2016-2021):
  Long  : 0
  Short : 17

Long signals:
Empty DataFrame
Columns: [event_type, ticker, entry_offset, exit_offset, n, avg_ret, med_ret, win_rate]
Index: []

Short signals:
  event_type ticker  entry_offset  exit_offset  n avg_ret med_ret win_rate
         CPI    GLD            -3            2 74 -11.91% -21.61%      20%
         CPI    GLD            -3            3 70  -8.86% -17.77%      26%
         CPI    GLD            -3            1 74  -8.81% -17.41%      26%
         CPI    IWM            -2            3 14  -8.77%  -9.87%      29%
         CPI    XLK            -3            2 19  -8.44% -13.63%      42%
         CPI    SPY            -3            3 12  -7.74%  -9.52%      25%
Unemployment    GLD            -3     